# Notebook 7: CUPED - Variance Reduction for A/B Testing

**Controlled-experiment Using Pre-Experiment Data**

## Overview

CUPED is a powerful statistical technique that uses historical customer data to reduce the variance of treatment effect estimates. In simpler terms: if we already know something about customers before the experiment (like their historical spending), we can use that information to get more precise estimates of how the email campaign affected them. Less noise means we can detect smaller effects and need fewer customers in our tests.

In [1]:
import os
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import seaborn as sns

# Optional plotly for interactive viz
try:
    import plotly.graph_objects as go
    import plotly.express as px
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Create output directory
os.makedirs('../data/outputs/nb07', exist_ok=True)


In [2]:
# Load the cleaned Hillstrom dataset
df = pd.read_csv('../data/outputs/nb01/nb01_hillstrom_clean.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nTreatment group counts:")
print(df['segment'].value_counts())
print(f"\nBasic statistics:")
print(df[['visit', 'conversion', 'spend', 'history', 'recency']].describe())

Dataset shape: (64000, 24)

Columns: ['recency', 'history_segment', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel', 'segment', 'visit', 'conversion', 'spend', 'email_match', 'email_match_simple', 'treatment', 'buyer_type', 'cross_shopper', 'zip_code_encoded', 'channel_encoded', 'history_log', 'spending_velocity', 'high_value', 'recency_segment', 'rfm_score']

First few rows:
   recency history_segment  history  mens  womens   zip_code  newbie channel  \
0       10  2) $100 - $200   142.44     1       0  Surburban       0   Phone   
1        6  3) $200 - $350   329.08     1       1      Rural       1     Web   
2        7  2) $100 - $200   180.65     0       1  Surburban       1     Web   
3        9  5) $500 - $750   675.83     1       0      Rural       1     Web   
4        2    1) $0 - $100    45.34     1       0      Urban       0     Web   

         segment  visit  ...  treatment   buyer_type cross_shopper  \
0  Womens E-Mail      0  ...          1    mens_only     

## Concept: Why Variance Reduction Matters

When we run an A/B test, we compare the average outcome between groups. But there's always some **noise** (randomness) in customer behavior. Some customers naturally spend more or less based on their history, not because of the email.

**The Problem:** If we just look at raw spending, this natural variation (noise) makes it hard to see the true effect of the email.

**The Solution:** CUPED says: "We already know how much each customer spent historically. Let's adjust for that." By removing the predictable part of customer behavior (based on history), we can see the email's effect more clearly.

**Why This Matters:**
- **Better precision:** Narrower confidence intervals
- **Faster detection:** Can spot real effects with fewer users
- **Cost savings:** Need fewer experiment participants for the same statistical power

### The Math (in Plain Terms)

Instead of comparing raw spend Y, we compare an **adjusted** version:

```
Y_adjusted = Y - θ × (History - Average_History)
```

Where:
- Y = actual customer spending in experiment
- History = customer's historical spending
- θ = a number we calculate that tells us how much historical spending predicts experimental spending
- Average_History = mean historical spending across all customers

The θ is calculated as:
```
θ = Correlation_Strength / How_Much_History_Varies
```

A stronger correlation and less variation in history means a bigger adjustment and more variance reduction.

## The CUPED Method

CUPED (Controlled-experiment Using Pre-Experiment Data) has these key steps:

1. **Choose a covariate** (X): a pre-experiment variable we know predicts the outcome (e.g., historical spending)
2. **Estimate theta (θ)**: quantify how predictive X is for the outcome Y
3. **Adjust the outcome**: Create Y_cuped = Y - θ × (X - E[X])
4. **Run your test** on the adjusted outcome instead of the raw outcome
5. **Calculate variance reduction**: Compare var(Y) vs var(Y_cuped) to see the improvement

### Why It Works

The adjusted outcome has the same treatment effect (because we subtract the same thing from treatment AND control groups), but lower variance (because we removed predictable noise).

In [3]:
# Step 1: Prepare data for CUPED on spend outcome
# We'll use 'history' (historical spending) as our pre-experiment covariate

# Separate treatment and control (use Mens Email vs No Email)
men_email = df[df['segment'] == 'Mens E-Mail'].copy()
control = df[df['segment'] == 'No E-Mail'].copy()

# Combine for full analysis
treatment_data = pd.concat([men_email, control]).reset_index(drop=True)

print(f"Mens Email sample size: {len(men_email)}")
print(f"Control sample size: {len(control)}")

# Calculate theta: Cov(spend, history) / Var(history)
# We use the control group to estimate the relationship
X_control = control['history'].values
Y_control = control['spend'].values

# Calculate covariance and variance
covariance_xy = np.cov(X_control, Y_control)[0, 1]
variance_x = np.var(X_control, ddof=1)
theta = covariance_xy / variance_x

print(f"\nCovariance(spend, history): {covariance_xy:.4f}")
print(f"Variance(history): {variance_x:.4f}")
print(f"Theta: {theta:.6f}")
print(f"\nInterpretation: For every dollar of history difference,")
print(f"we expect ~${theta:.4f} difference in experimental spend")

Mens Email sample size: 21307
Control sample size: 21306

Covariance(spend, history): 43.0724
Variance(history): 63877.1850
Theta: 0.000674

Interpretation: For every dollar of history difference,
we expect ~$0.0007 difference in experimental spend


In [4]:
# Step 2: Create CUPED-adjusted spend metric
mean_history = treatment_data['history'].mean()

# Calculate adjusted spend for all observations
treatment_data['spend_cuped'] = (treatment_data['spend'] - 
                                  theta * (treatment_data['history'] - mean_history))

# Calculate variances
var_raw_spend = treatment_data.groupby('segment')['spend'].var()
var_cuped_spend = treatment_data.groupby('segment')['spend_cuped'].var()

print("Variance Comparison for Spend:")
print("\nRaw Spend Variance:")
print(var_raw_spend)
print("\nCUPED-Adjusted Spend Variance:")
print(var_cuped_spend)

# Calculate variance reduction percentage
var_reduction = (1 - var_cuped_spend / var_raw_spend) * 100
print("\nVariance Reduction (%):")
print(var_reduction)

# The treatment effect should be the same
men_email_idx = treatment_data['segment'] == 'Mens E-Mail'
control_idx = treatment_data['segment'] == 'No E-Mail'

effect_raw = treatment_data[men_email_idx]['spend'].mean() - treatment_data[control_idx]['spend'].mean()
effect_cuped = treatment_data[men_email_idx]['spend_cuped'].mean() - treatment_data[control_idx]['spend_cuped'].mean()

print(f"\nTreatment Effect on Raw Spend: ${effect_raw:.2f}")
print(f"Treatment Effect on CUPED Spend: ${effect_cuped:.2f}")
print(f"(Effect is the same, but variance is lower)")

Variance Comparison for Spend:

Raw Spend Variance:
segment
Mens E-Mail    315.211799
No E-Mail      134.286372
Name: spend, dtype: float64

CUPED-Adjusted Spend Variance:
segment
Mens E-Mail    315.084583
No E-Mail      134.257328
Name: spend_cuped, dtype: float64

Variance Reduction (%):
segment
Mens E-Mail    0.040359
No E-Mail      0.021628
dtype: float64

Treatment Effect on Raw Spend: $0.77
Treatment Effect on CUPED Spend: $0.77
(Effect is the same, but variance is lower)


In [5]:
# Step 3: Run t-test on CUPED-adjusted spend
from scipy.stats import ttest_ind, norm

treatment_spend_cuped = treatment_data[men_email_idx]['spend_cuped']
control_spend_cuped = treatment_data[control_idx]['spend_cuped']

# Two-sample t-test
t_stat, p_value = ttest_ind(treatment_spend_cuped, control_spend_cuped)

# Calculate confidence intervals
n_treat = len(treatment_spend_cuped)
n_control = len(control_spend_cuped)
mean_treat = treatment_spend_cuped.mean()
mean_control = control_spend_cuped.mean()
std_treat = treatment_spend_cuped.std(ddof=1)
std_control = control_spend_cuped.std(ddof=1)
se_diff = np.sqrt(std_treat**2/n_treat + std_control**2/n_control)
df_welch = (std_treat**2/n_treat + std_control**2/n_control)**2 / ((std_treat**2/n_treat)**2/(n_treat-1) + (std_control**2/n_control)**2/(n_control-1))
t_crit = stats.t.ppf(0.975, df_welch)
ci_lower = effect_cuped - t_crit * se_diff
ci_upper = effect_cuped + t_crit * se_diff

print("\n=== T-Test on CUPED-Adjusted Spend ===")
print(f"Mens Email Mean (CUPED): ${mean_treat:.2f}")
print(f"Control Mean (CUPED): ${mean_control:.2f}")
print(f"Effect Size: ${effect_cuped:.2f}")
print(f"95% CI: [${ci_lower:.2f}, ${ci_upper:.2f}]")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Significant at α=0.05? {p_value < 0.05}")


=== T-Test on CUPED-Adjusted Spend ===
Mens Email Mean (CUPED): $1.42
Control Mean (CUPED): $0.65
Effect Size: $0.77
95% CI: [$0.48, $1.05]
T-statistic: 5.2919
P-value: 0.0000
Significant at α=0.05? True


In [6]:
# (Chart cell removed — interactive Plotly version rendered below in the "Blog-Ready Plotly Charts" section.)

In [7]:
# Step 5: Apply CUPED to binary outcome (visit) using recency
# For binary outcomes, we still use the same approach

X_control_recency = control['recency'].values
Y_control_visit = control['visit'].astype(float).values

covariance_visit = np.cov(X_control_recency, Y_control_visit)[0, 1]
variance_recency = np.var(X_control_recency, ddof=1)
theta_visit = covariance_visit / variance_recency

print("\n=== CUPED on Visit (Binary Outcome) ===")
print(f"Using 'recency' as covariate")
print(f"Theta: {theta_visit:.6f}")

# Create adjusted visit metric
mean_recency = treatment_data['recency'].mean()
treatment_data['visit_cuped'] = (treatment_data['visit'].astype(float) - 
                                   theta_visit * (treatment_data['recency'] - mean_recency))

# Compare variance
var_raw_visit = treatment_data.groupby('segment')['visit'].var()
var_cuped_visit = treatment_data.groupby('segment')['visit_cuped'].var()

var_reduction_visit = (1 - var_cuped_visit / var_raw_visit) * 100

print("\nVariance Reduction for Visit:")
print(f"Raw Visit Variance: {var_raw_visit.mean():.6f}")
print(f"CUPED Visit Variance: {var_cuped_visit.mean():.6f}")
print(f"Variance Reduction: {var_reduction_visit.mean():.2f}%")

# Test on adjusted visit
treatment_visit_cuped = treatment_data[men_email_idx]['visit_cuped']
control_visit_cuped = treatment_data[control_idx]['visit_cuped']

t_stat_visit, p_value_visit = ttest_ind(treatment_visit_cuped, control_visit_cuped)
effect_cuped_visit = treatment_visit_cuped.mean() - control_visit_cuped.mean()

print(f"\nEffect on CUPED Visit: {effect_cuped_visit:.4f}")
print(f"P-value: {p_value_visit:.4f}")
print(f"Significant? {p_value_visit < 0.05}")


=== CUPED on Visit (Binary Outcome) ===
Using 'recency' as covariate
Theta: -0.007320

Variance Reduction for Visit:
Raw Visit Variance: 0.122132
CUPED Visit Variance: 0.121339
Variance Reduction: 0.66%

Effect on CUPED Visit: 0.0768
P-value: 0.0000
Significant? True


## Multi-Covariate CUPED Using Regression

Instead of using just one covariate, we can use multiple pre-experiment variables in a linear regression model. This is more powerful when multiple factors predict the outcome.

The approach:
1. Train a linear regression on the control group: spend ~ history + recency + mens + womens + newbie
2. Predict spend for all subjects using this model
3. Create adjusted metric: Y_cuped = Y - (Y_predicted - Mean_Y_predicted)
4. Run analysis on adjusted metric

This captures more variance reduction because multiple variables together predict outcomes better than any single variable.

In [8]:
# Step 6: Multi-covariate CUPED using regression
from sklearn.preprocessing import StandardScaler

# Prepare features for regression
features = ['history', 'recency', 'mens', 'womens', 'newbie']

# Ensure features exist and are numeric
for feat in features:
    if feat not in treatment_data.columns:
        print(f"Warning: {feat} not in dataframe")
    
X_features = treatment_data[features].copy()
y_spend = treatment_data['spend'].copy()

# Standardize features for better interpretation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

# Train regression model on control group to estimate relationship
control_mask = treatment_data['segment'] == 'No E-Mail'
X_control_reg = X_scaled[control_mask]
y_control_spend = y_spend[control_mask]

model = LinearRegression()
model.fit(X_control_reg, y_control_spend)

# Predict for all subjects
y_pred_all = model.predict(X_scaled)
y_pred_mean = y_pred_all.mean()

# Create multi-covariate adjusted spend
treatment_data['spend_cuped_multi'] = y_spend - (y_pred_all - y_pred_mean)

# Compare variance reduction
var_raw = treatment_data['spend'].var()
var_cuped_single = treatment_data['spend_cuped'].var()
var_cuped_multi = treatment_data['spend_cuped_multi'].var()

print("=== Variance Reduction Comparison ===")
print(f"Raw Spend Variance: {var_raw:.2f}")
print(f"Single Covariate CUPED Variance: {var_cuped_single:.2f}")
print(f"Multi-Covariate CUPED Variance: {var_cuped_multi:.2f}")
print(f"\nReduction (Single): {(1 - var_cuped_single/var_raw)*100:.2f}%")
print(f"Reduction (Multi): {(1 - var_cuped_multi/var_raw)*100:.2f}%")

# Test effect with multi-covariate CUPED
treatment_spend_cuped_multi = treatment_data[men_email_idx]['spend_cuped_multi']
control_spend_cuped_multi = treatment_data[control_idx]['spend_cuped_multi']

t_stat_multi, p_value_multi = ttest_ind(treatment_spend_cuped_multi, control_spend_cuped_multi)
effect_cuped_multi = treatment_spend_cuped_multi.mean() - control_spend_cuped_multi.mean()

print(f"\nEffect (Multi-Covariate CUPED): ${effect_cuped_multi:.2f}")
print(f"P-value: {p_value_multi:.4f}")

print(f"\nRegression Model R²: {model.score(X_control_reg, y_control_spend):.4f}")
print(f"Feature Coefficients (standardized):")
for feat, coef in zip(features, model.coef_):
    print(f"  {feat}: {coef:.4f}")

=== Variance Reduction Comparison ===
Raw Spend Variance: 224.89
Single Covariate CUPED Variance: 224.82
Multi-Covariate CUPED Variance: 224.66

Reduction (Single): 0.03%
Reduction (Multi): 0.10%

Effect (Multi-Covariate CUPED): $0.77
P-value: 0.0000

Regression Model R²: 0.0014
Feature Coefficients (standardized):
  history: 0.1636
  recency: -0.1553
  mens: 0.2544
  womens: 0.1514
  newbie: -0.3358


## Variance Reduction Quantification

By reducing variance, CUPED allows us to detect smaller effects or use fewer sample sizes. Here's how:

**Statistical Power:** The ability to detect a true effect if it exists
- Larger variance = lower power (harder to detect effects)
- Smaller variance = higher power (easier to detect effects)

**Sample Size Calculation:** 
For a fixed power level (e.g., 80%), sample size is inversely related to variance reduction.

```
n_needed_after_cuped ≈ n_needed_before × (variance_after / variance_before)
```

**Confidence Interval Width:**
CI width is proportional to sqrt(variance). Smaller variance = tighter CIs.

In [9]:
# Step 7: Quantify how much sample size is reduced
from scipy.stats import norm

# Assume we want 80% power to detect a $20 effect with alpha=0.05 (two-tailed)
target_effect = 20
alpha = 0.05
power = 0.80

# For t-test, approximate sample size with:
# n = 2 * sigma^2 * (z_alpha + z_beta)^2 / effect^2

z_alpha = norm.ppf(1 - alpha/2)  # ~1.96
z_beta = norm.ppf(power)  # ~0.84

# Raw spend variance
var_spend_raw = treatment_data['spend'].var()
n_raw = 2 * var_spend_raw * (z_alpha + z_beta)**2 / (target_effect**2)

# CUPED (single) variance
n_cuped_single = 2 * var_cuped_single * (z_alpha + z_beta)**2 / (target_effect**2)

# CUPED (multi) variance
n_cuped_multi = 2 * var_cuped_multi * (z_alpha + z_beta)**2 / (target_effect**2)

print("=== Sample Size to Detect $20 Effect (80% Power) ===")
print(f"Raw Spend Approach: {n_raw:.0f} customers per group ({n_raw*2:.0f} total)")
print(f"Single-Cov CUPED: {n_cuped_single:.0f} customers per group ({n_cuped_single*2:.0f} total)")
print(f"Multi-Cov CUPED: {n_cuped_multi:.0f} customers per group ({n_cuped_multi*2:.0f} total)")

savings_single = (1 - n_cuped_single/n_raw) * 100
savings_multi = (1 - n_cuped_multi/n_raw) * 100

print(f"\nSample Size Reduction:")
print(f"Single-Cov CUPED: {savings_single:.1f}% fewer customers needed")
print(f"Multi-Cov CUPED: {savings_multi:.1f}% fewer customers needed")

# Visualize CI width reduction
ci_width_raw = 1.96 * 2 * np.sqrt(var_spend_raw / (len(men_email)/2))
ci_width_cuped = 1.96 * 2 * np.sqrt(var_cuped_single / (len(men_email)/2))
ci_width_cuped_multi = 1.96 * 2 * np.sqrt(var_cuped_multi / (len(men_email)/2))

print(f"\n95% CI Width (assuming current sample size):")
print(f"Raw Spend: ±${ci_width_raw/2:.2f}")
print(f"CUPED (Single): ±${ci_width_cuped/2:.2f}")
print(f"CUPED (Multi): ±${ci_width_cuped_multi/2:.2f}")

=== Sample Size to Detect $20 Effect (80% Power) ===
Raw Spend Approach: 9 customers per group (18 total)
Single-Cov CUPED: 9 customers per group (18 total)
Multi-Cov CUPED: 9 customers per group (18 total)

Sample Size Reduction:
Single-Cov CUPED: 0.0% fewer customers needed
Multi-Cov CUPED: 0.1% fewer customers needed

95% CI Width (assuming current sample size):
Raw Spend: ±$0.28
CUPED (Single): ±$0.28
CUPED (Multi): ±$0.28


In [10]:
# (Chart cell removed — interactive Plotly version rendered below in the "Blog-Ready Plotly Charts" section.)

## Comparison: Naive vs CUPED Results

Here we create a side-by-side comparison of all our methods to see how CUPED improves our statistical power.

In [11]:
# Step 8: Comprehensive comparison table
comparison_results = []

# Raw spend test
t_stat_raw, p_value_raw = ttest_ind(
    treatment_data[men_email_idx]['spend'],
    treatment_data[control_idx]['spend']
)
effect_raw_spend = (treatment_data[men_email_idx]['spend'].mean() - 
                    treatment_data[control_idx]['spend'].mean())
se_raw = np.sqrt(var_spend_raw * (2/len(men_email)))
ci_raw_spend = (effect_raw_spend - 1.96*se_raw, effect_raw_spend + 1.96*se_raw)

comparison_results.append({
    'Method': 'Raw Spend',
    'Effect': f'${effect_raw_spend:.2f}',
    'SE': f'${se_raw:.2f}',
    'CI_Lower': f'${ci_raw_spend[0]:.2f}',
    'CI_Upper': f'${ci_raw_spend[1]:.2f}',
    'P-Value': f'{p_value_raw:.4f}',
    'Significant': 'Yes' if p_value_raw < 0.05 else 'No',
    'Variance': f'{var_spend_raw:.2f}'
})

# CUPED (single covariate)
se_cuped = np.sqrt(var_cuped_single * (2/len(men_email)))
ci_cuped_spend = (effect_cuped - 1.96*se_cuped, effect_cuped + 1.96*se_cuped)

comparison_results.append({
    'Method': 'CUPED (History)',
    'Effect': f'${effect_cuped:.2f}',
    'SE': f'${se_cuped:.2f}',
    'CI_Lower': f'${ci_cuped_spend[0]:.2f}',
    'CI_Upper': f'${ci_cuped_spend[1]:.2f}',
    'P-Value': f'{p_value:.4f}',
    'Significant': 'Yes' if p_value < 0.05 else 'No',
    'Variance': f'{var_cuped_single:.2f}'
})

# CUPED (multi-covariate)
se_cuped_multi = np.sqrt(var_cuped_multi * (2/len(men_email)))
ci_cuped_multi_spend = (effect_cuped_multi - 1.96*se_cuped_multi, effect_cuped_multi + 1.96*se_cuped_multi)

comparison_results.append({
    'Method': 'CUPED (Multi)',
    'Effect': f'${effect_cuped_multi:.2f}',
    'SE': f'${se_cuped_multi:.2f}',
    'CI_Lower': f'${ci_cuped_multi_spend[0]:.2f}',
    'CI_Upper': f'${ci_cuped_multi_spend[1]:.2f}',
    'P-Value': f'{p_value_multi:.4f}',
    'Significant': 'Yes' if p_value_multi < 0.05 else 'No',
    'Variance': f'{var_cuped_multi:.2f}'
})

comparison_df = pd.DataFrame(comparison_results)
print("\n=== COMPREHENSIVE RESULTS COMPARISON ===\n")
print(comparison_df.to_string(index=False))

# Save results
comparison_df.to_csv('../data/outputs/nb07/nb07_cuped_results.csv', index=False)
print("\nResults saved to: ../data/outputs/nb07/nb07_cuped_results.csv")


=== COMPREHENSIVE RESULTS COMPARISON ===

         Method Effect    SE CI_Lower CI_Upper P-Value Significant Variance
      Raw Spend  $0.77 $0.15    $0.49    $1.05  0.0000         Yes   224.89
CUPED (History)  $0.77 $0.15    $0.48    $1.05  0.0000         Yes   224.82
  CUPED (Multi)  $0.77 $0.15    $0.48    $1.05  0.0000         Yes   224.66

Results saved to: ../data/outputs/nb07/nb07_cuped_results.csv


## Key Takeaways

1. **Variance Reduction Works**: CUPED reduces variance by leveraging pre-experiment data
2. **Same Effect Estimate**: The treatment effect doesn't change, but precision improves
3. **Practical Benefits**: 
   - Narrower confidence intervals
   - Fewer customers needed for same power
   - Ability to detect smaller effects
4. **Multi-Covariate is Best**: Using multiple predictors (history, recency, etc.) provides the most variance reduction
5. **When to Use CUPED**: 
   - When you have good pre-experiment data
   - When outcome variance is high
   - When you want to detect small effects efficiently

## Next Steps

- Consider CUPED for all future experiments
- Choose covariates based on their predictive power
- Validate results with out-of-sample data
- Document baseline metrics for future experiments

---

## Blog-Ready Plotly Charts

The cells below regenerate the charts from this notebook as responsive Plotly
HTML files for embedding in the blog post. They are **self-contained**: each
one re-loads the clean dataset from nb01 and re-derives the statistics it
needs, so you can run this section in isolation.

Outputs are written to `data/outputs/nb##/` with the suffix `_interactive.html`.

**Required packages:** `plotly` (install with `pip install plotly` if missing).

In [12]:
# ============================================================
# Blog-Ready Plotly Charts — self-contained, embed-friendly
# ============================================================
# These cells produce responsive Plotly HTML files for the blog post.
# They re-load from the nb01 clean CSV and re-derive stats so the section
# runs standalone. Each figure uses:
#   - include_plotlyjs='cdn' (single shared CDN load on the blog page)
#   - config={'responsive': True} so it resizes to container width
#   - automargin=True on axes + generous margins so labels never clip
#   - rotated tick labels on long categories, headroom for outside labels
import os, numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "notebook_connected"  # inline-render Plotly in cell output

OUT_DIR = os.path.abspath("../data/outputs/nb07")
os.makedirs(OUT_DIR, exist_ok=True)
CLEAN_CSV = os.path.abspath("../data/outputs/nb01/nb01_hillstrom_clean.csv")
df_blog = pd.read_csv(CLEAN_CSV)

# Shared palette aligned with nb01 Plotly charts
COLORS = {
    "Mens E-Mail": "#4C8BB8", "Womens E-Mail": "#5FA85F", "No E-Mail": "#E89B4C",
    "Match": "#2ECC71", "Mismatch": "#E74C3C", "Mixed": "#F39C12", "Control": "#95A5A6",
    "Treatment (Any Email)": "#4C8BB8",
}
PLOTLY_KW = dict(include_plotlyjs="cdn", full_html=True,
                 config={"responsive": True, "displaylogo": False})
BASE_LAYOUT = dict(template="plotly_white",
                   font=dict(family="Arial, sans-serif", size=13),
                   title_x=0.5,
                   margin=dict(l=70, r=40, t=90, b=90),
                   hoverlabel=dict(bgcolor="white", font_size=12))
print(f"Blog-ready Plotly charts will be written to: {OUT_DIR}")


Blog-ready Plotly charts will be written to: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/ab_testing/data/outputs/nb07


In [14]:
from scipy import stats as spstats
# CUPED using pre-period covariate `history` as the pre-experiment covariate
y = df_blog["spend"].values
X = df_blog["history"].values
theta = np.cov(y, X, ddof=1)[0,1] / np.var(X, ddof=1)
y_cuped = y - theta * (X - X.mean())
var_ratio = np.var(y_cuped, ddof=1) / np.var(y, ddof=1)
pct_reduction = (1 - var_ratio) * 100

# Chart 1: Variance reduction bar
fig = go.Figure()
fig.add_trace(go.Bar(x=["Unadjusted Spend", "CUPED-Adjusted Spend"],
                     y=[np.var(y, ddof=1), np.var(y_cuped, ddof=1)],
                     marker_color=["#95A5A6", "#4C8BB8"],
                     text=[f"σ² = {np.var(y, ddof=1):.2f}", f"σ² = {np.var(y_cuped, ddof=1):.2f}"],
                     textposition="outside",
                     hovertemplate="<b>%{x}</b><br>Variance: %{y:.3f}<extra></extra>"))
fig.update_layout(**BASE_LAYOUT,
                  title=f"CUPED Variance Reduction — {pct_reduction:.1f}% reduction<br>"
                        f"<sub>θ = {theta:.4f}, pre-covariate = history</sub>",
                  xaxis=dict(automargin=True),
                  yaxis=dict(title="Variance of Spend", automargin=True,
                             range=[0, max(np.var(y, ddof=1), np.var(y_cuped, ddof=1))*1.15]),
                  height=460)
fig.write_html(os.path.join(OUT_DIR, "nb07_variance_reduction_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb07_variance_reduction_interactive.html")

# Chart 2: Pre vs Post (history vs spend) scatter — illustrates CUPED's stability assumption
samp = df_blog.sample(min(3000, len(df_blog)), random_state=42)
fig = go.Figure(go.Scatter(x=samp["history"], y=samp["spend"], mode="markers",
                           marker=dict(color=samp["treatment"].map({0:"#95A5A6",1:"#4C8BB8"}),
                                       size=6, opacity=0.55,
                                       line=dict(width=0.3, color="black")),
                           hovertemplate="History: $%{x:.2f}<br>Spend: $%{y:.2f}<extra></extra>",
                           showlegend=False))
# Overlay regression lines by treatment
for t_val, color, name in [(0, "#E89B4C", "Control fit"), (1, "#4C8BB8", "Treated fit")]:
    sub = samp[samp["treatment"] == t_val]
    if len(sub) > 5:
        m_, b_ = np.polyfit(sub["history"], sub["spend"], 1)
        xs = np.linspace(sub["history"].min(), sub["history"].max(), 50)
        fig.add_trace(go.Scatter(x=xs, y=m_*xs+b_, mode="lines",
                                 line=dict(color=color, width=3), name=name))
fig.update_layout(**BASE_LAYOUT,
                  title="Pre-Period History vs Post-Period Spend — CUPED Correlation",
                  xaxis=dict(title="Pre-period purchase history ($)", automargin=True),
                  yaxis=dict(title="Post-period spend ($)", automargin=True),
                  height=520,
                  legend=dict(orientation="h", y=-0.2, x=0.5, xanchor="center"))
fig.write_html(os.path.join(OUT_DIR, "nb07_pre_post_scatter_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb07_pre_post_scatter_interactive.html")

# Chart 3: Power curve — standard vs CUPED at same sample size
baseline_var = np.var(y, ddof=1)
cuped_var = np.var(y_cuped, ddof=1)
ns = np.linspace(500, 25000, 60)
# Minimum detectable mean difference at power 0.8, alpha 0.05
z_a = spstats.norm.ppf(0.975); z_b = spstats.norm.ppf(0.80)
mde_std = (z_a + z_b) * np.sqrt(2 * baseline_var / ns)
mde_cup = (z_a + z_b) * np.sqrt(2 * cuped_var / ns)
fig = go.Figure()
fig.add_trace(go.Scatter(x=ns, y=mde_std, mode="lines", name="Standard t-test",
                         line=dict(color="#95A5A6", width=3)))
fig.add_trace(go.Scatter(x=ns, y=mde_cup, mode="lines", name="CUPED-adjusted",
                         line=dict(color="#4C8BB8", width=3)))
fig.update_layout(**BASE_LAYOUT,
                  title=f"Power Curve — MDE for Spend (Standard vs CUPED)<br>"
                        f"<sub>CUPED reduces required sample size by ~{pct_reduction:.1f}%</sub>",
                  xaxis=dict(title="Sample size per arm", automargin=True),
                  yaxis=dict(title="Min Detectable Spend Difference ($)", automargin=True),
                  height=500)
fig.write_html(os.path.join(OUT_DIR, "nb07_power_comparison_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb07_power_comparison_interactive.html")


  ✓ nb07_variance_reduction_interactive.html


  ✓ nb07_pre_post_scatter_interactive.html


  ✓ nb07_power_comparison_interactive.html


### Results-Display Tables (embed-ready go.Table cards for every printed output)

Each card below mirrors one of the printed console blocks and saves as its own
HTML file under `../data/outputs/nb07/` so it can be dropped straight into the
blog post.


In [15]:
# Reusable go.Table card helper (reuses OUT_DIR/PLOTLY_KW/BASE_LAYOUT from Plotly setup cell above)
def table_card(title, header_vals, cell_cols, colwidths,
               row_colors=None, cell_font_size=12, height_extra=80, align="center"):
    n_rows = len(cell_cols[0]) if cell_cols else 0
    stripe = ["#F8F9F9" if i%2==0 else "white" for i in range(n_rows)]
    fill = row_colors if row_colors else [stripe for _ in cell_cols]
    fig = go.Figure(data=[go.Table(
        columnwidth=colwidths,
        header=dict(values=[f"<b>{h}</b>" for h in header_vals],
                    fill_color="#2C3E50",
                    font=dict(color="white", size=13),
                    align="center", height=36),
        cells=dict(values=cell_cols, fill_color=fill, align=align,
                   font=dict(size=cell_font_size, family="monospace"),
                   height=30))])
    fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=20, r=20, t=60, b=20)},
                      title=title,
                      height=36 + 30*n_rows + height_extra)
    return fig

def sig_colors(flags):
    return ["#D5F5E3" if f else "#FADBD8" for f in flags]

def stripe_col(n):
    return ["#F8F9F9" if i%2==0 else "white" for i in range(n)]

# CUPED Variance Reduction — Results-Display Table
# Use `history` (pre-period covariate) to adjust post-period outcome
from scipy import stats as _sp

any_email = df_blog[df_blog["segment"] != "No E-Mail"].copy()
control   = df_blog[df_blog["segment"] == "No E-Mail"].copy()

def cuped_adjust(y, x):
    x_mean = x.mean()
    theta = np.cov(y, x, ddof=1)[0,1] / np.var(x, ddof=1) if np.var(x, ddof=1) > 0 else 0
    y_adj = y - theta*(x - x_mean)
    return y_adj, theta

def compare(outcome):
    a = any_email[outcome].values; ca = control[outcome].values
    h_a = any_email["history"].values; h_c = control["history"].values
    # pool both arms for theta estimation (standard CUPED)
    y_all = np.concatenate([a, ca])
    x_all = np.concatenate([h_a, h_c])
    y_adj, theta = cuped_adjust(y_all, x_all)
    a_adj = y_adj[:len(a)]; ca_adj = y_adj[len(a):]
    # Raw
    d_raw = a.mean() - ca.mean()
    se_raw = np.sqrt(a.var(ddof=1)/len(a) + ca.var(ddof=1)/len(ca))
    # Adjusted
    d_adj = a_adj.mean() - ca_adj.mean()
    se_adj = np.sqrt(a_adj.var(ddof=1)/len(a_adj) + ca_adj.var(ddof=1)/len(ca_adj))
    var_reduction = 1 - (se_adj**2)/(se_raw**2)
    return dict(theta=theta, d_raw=d_raw, se_raw=se_raw,
                d_adj=d_adj, se_adj=se_adj, var_reduction=var_reduction,
                t_raw=d_raw/se_raw if se_raw>0 else 0,
                t_adj=d_adj/se_adj if se_adj>0 else 0)

results = {o: compare(o) for o in ["visit", "conversion", "spend"]}

rows = []
for outcome, r in results.items():
    rows.append((outcome.capitalize(),
                 f"{r['theta']:.5f}",
                 f"{r['d_raw']:+.5f}",
                 f"{r['se_raw']:.5f}",
                 f"{r['t_raw']:.3f}",
                 f"{r['d_adj']:+.5f}",
                 f"{r['se_adj']:.5f}",
                 f"{r['t_adj']:.3f}",
                 f"{r['var_reduction']*100:.1f}%"))

fig = go.Figure(data=[go.Table(
    columnwidth=[130, 110, 130, 120, 100, 140, 120, 100, 130],
    header=dict(values=[f"<b>{h}</b>" for h in
                        ["Outcome","θ (CUPED)","Raw diff","Raw SE","Raw t",
                         "Adjusted diff","Adjusted SE","Adjusted t","Variance ↓"]],
                fill_color="#2C3E50", font=dict(color="white", size=13), align="center", height=36),
    cells=dict(values=list(zip(*rows)), fill_color=[stripe_col(len(rows))]*9,
               align="center", font=dict(size=12, family="monospace"), height=30))])
fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=20, r=20, t=60, b=20)},
                  title="CUPED Adjustment — Variance Reduction via `history` Pre-Period Covariate",
                  height=36 + 30*len(rows) + 100)
fig.write_html(os.path.join(OUT_DIR, "nb07_cuped_comparison_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb07 CUPED comparison card saved")


  ✓ nb07 CUPED comparison card saved
